An **n-gram score** is a metric that measures how well a language model (like the one that predicts text on your phone) predicts a sequence of words. It does this by checking how often a specific n-gram appears in a given body of text. Essentially, the score is a way to evaluate the "predictive power" of the model.

Think of it like this: if a language model sees the phrase "I went to the store and bought..." it will look at all the words that follow "bought" in its training data. If "milk" is a very common word to follow "bought," the model gives it a high probability, or a high "n-gram score." The higher the score for a specific n-gram, the more likely it is to appear in a natural sentence.

In [ ]:
from collections import Counter
import pandas as pd

# Creating a sample time series dataset (daily stock prices)
data = {'Date': pd.to_datetime(['2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04', '2025-01-05', '2025-01-06', '2025-01-07', '2025-01-08', '2025-01-09', '2025-01-10']),
        'Price': [100, 102, 101, 103, 105, 104, 106, 105, 104, 106]}
df = pd.DataFrame(data)

# Convert prices to a list
prices = df['Price'].tolist()

def generate_ngrams(data, n):
    """
    Generates n-grams from a list of data points.
    """
    ngrams = []
    # Loop from the first element up to the point where the window fits
    for i in range(len(data) - n + 1):
        # Slice the list to get a contiguous sequence of n items
        ngrams.append(tuple(data[i:i + n]))
    return ngrams

def calculate_ngram_scores(data, n):
    """
    Calculates the frequency (score) of each n-gram.
    """
    # Generate the n-grams
    ngrams = generate_ngrams(data, n)
    # Count the frequency of each n-gram
    ngram_scores = Counter(ngrams)
    return ngram_scores

# Let's generate and score 3-grams
trigram_scores = calculate_ngram_scores(prices, 3)

print("Original Data:", prices)
print("\n3-gram Scores (Frequency):")
for trigram, score in trigram_scores.items():
    print(f"{trigram}: {score}")

Original Data: [100, 102, 101, 103, 105, 104, 106, 105, 104, 106]

3-gram Scores (Frequency):
(100, 102, 101): 1
(102, 101, 103): 1
(101, 103, 105): 1
(103, 105, 104): 1
(105, 104, 106): 2
(104, 106, 105): 1
(106, 105, 104): 1


# Assignment # 1

Change the n value: Modify the calculate_ngram_scores function call to use n=2 (bigrams) and then n=4. What happens to the scores? Why do you think some scores might decrease as n gets larger?

In [ ]:
data = {'Date': pd.to_datetime(['2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04', '2025-01-05', '2025-01-06', '2025-01-07', '2025-01-08', '2025-01-09', '2025-01-10']),
        'Price': [100, 102, 101, 103, 105, 104, 106, 105, 104, 106]}
df = pd.DataFrame(data)

prices = df['Price'].tolist()

def generate_ngrams(data, n):
    ngrams = []
    for i in range(len(data) - n + 1):
        ngrams.append(tuple(data[i:i + n]))
    return ngrams

def calculate_ngram_scores(data, n):
    ngrams = generate_ngrams(data, n)
    ngram_scores = Counter(ngrams)
    return ngram_scores


bigram_scores = calculate_ngram_scores(prices, 2)

print("Original Data:", prices)
print("\n2-gram Scores (Frequency):")
for bigram, score in bigram_scores.items():
    print(f"{bigram}: {score}")

fourgram_scores = calculate_ngram_scores(prices, 4)

print("Original Data:", prices)
print("\n4-gram Scores (Frequency):")
for fourgram, score in fourgram_scores.items():
    print(f"{fourgram}: {score}")

print("""
The scores repeat more often in the bigram. There are no repeats in the 4-gram. The quantity of repeats decrease as n increases.
This happens because n looks at larger patterns. It's harder for something to repeat multiple times when the amount is greater (because the contigent probability on each previous datapoint stacks).
For example, if you look for a pattern of two temperatures, the probability is just the probability of the first one, and the probability of the second given the first. This is plausible.
However, for a pattern of 4, you need each successive probability (fourth given third, third given second, second given first, first) which becomes exponentially less plausible.
Therefore the scores decrease as n increases.
""")

Original Data: [100, 102, 101, 103, 105, 104, 106, 105, 104, 106]

2-gram Scores (Frequency):
(100, 102): 1
(102, 101): 1
(101, 103): 1
(103, 105): 1
(105, 104): 2
(104, 106): 2
(106, 105): 1
Original Data: [100, 102, 101, 103, 105, 104, 106, 105, 104, 106]

4-gram Scores (Frequency):
(100, 102, 101, 103): 1
(102, 101, 103, 105): 1
(101, 103, 105, 104): 1
(103, 105, 104, 106): 1
(105, 104, 106, 105): 1
(104, 106, 105, 104): 1
(106, 105, 104, 106): 1

The scores repeat more often in the bigram. There are no repeats in the 4-gram. The quantity of repeats decrease as n increases.
This happens because n looks at larger patterns. It's harder for something to repeat multiple times when the amount is greater (because the contigent probability on each previous datapoint stacks).
For example, if you look for a pattern of two temperatures, the probability is just the probability of the first one, and the probability of the second given the first. This is plausible.
However, for a pattern of 4, y

# Assignment # 2
Probability vs. Frequency: The code above uses a simple frequency count. For a more sophisticated n-gram score, you could calculate the conditional probability. For example, for a trigram like (101, 103, 105), the score could be P(105 | 101, 103), which is the probability of 105 appearing after 101 and 103. How would you modify the Python code to calculate this type of score? Hint: You'll need to count the frequency of the full n-gram and the frequency of the (n-1)-gram that precedes it.

In [ ]:
def calculate_sophisticated_ngram(data, n):
  #Count the ngrams
  ngrams = generate_ngrams(data, n)
  ngram_counts = Counter(ngrams)
  #Generate prefixes to determine conditional probability
  prefix_ngrams = generate_ngrams(data, n-1)
  prefix_counts = Counter(prefix_ngrams)
  #Store final probabilities
  conditional_scores = {}
  for ngram, count in ngram_counts.items():
    prefix = ngram[:-1]
    conditional_scores[ngram] = count / prefix_counts[prefix]

  return conditional_scores

conditional_bigrams = calculate_sophisticated_ngram(prices, 2)

print("Conditional bigram probabilities:")
for bigram, prob in conditional_bigrams.items():
    print(f"{bigram}: {prob:.2f}")

Conditional trigram probabilities:
(100, 102): 1.00
(102, 101): 1.00
(101, 103): 1.00
(103, 105): 1.00
(105, 104): 1.00
(104, 106): 1.00
(106, 105): 0.50


# Assignment # 3
Real-World Application: Imagine a dataset of daily average temperatures for your city over a few years. How could you use n-gram scores to find and predict seasonal patterns? For example, what would a high-scoring bigram (T1, T2) tell you about consecutive temperatures?

In [ ]:
"""
Ngrams would be useful for determining what another temperature could look like.
For example, if one bigram or trigram was high-scoring, it could predict what the temperature looks like tommorow.
If (95, 100) was a high-scoring bigram, if today is 95 degrees, I might prepare for even hotter temperatures tommorow.
This would be useful to meteorologists to predict the weather based on what it has been before.
"""